# Genesis AI — opportunistic Colab accelerator

Interactive-only free accelerator sidecar. The canonical 24/7 loop runs remotely on GitHub Actions; this notebook may validate a CPU-screened model candidate when a user is actively using Colab. It has no checkpoint-promotion authority.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys
REPO = pathlib.Path('/content/genesis-ai')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/NahuelGenchi/genesis-ai.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--force','origin/main'], check=True)
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','-e',str(REPO)], check=True)
os.chdir(REPO)


In [ ]:
from genesis_ai.accelerator_job import detect_accelerator
runtime = detect_accelerator('colab')
assert runtime['device'] == 'cuda', 'Choose Runtime > Change runtime type > GPU'
runtime


## Select the latest CPU-screened model candidate
The CPU farm persists its latest aggregate shortlist on `main`. This cell selects the strongest committed `tiny-model` manifest that survived that screen.

In [ ]:
summary_path = REPO / 'research/accelerators/cpu-farm-latest.json'
assert summary_path.is_file(), 'No persisted CPU-farm shortlist yet; let the scheduled farm complete first.'
summary = json.loads(summary_path.read_text())
eligible = {(x['lane'], x['variant']): float(x['improvement_fraction']) for x in summary['expensive_stage_eligible']}
choices = []
for path in sorted((REPO/'accelerators/jobs').glob('*.json')):
    job = json.loads(path.read_text())
    pair = (job.get('cpu_screen',{}).get('lane'), job.get('cpu_screen',{}).get('variant'))
    if job.get('enabled') is True and job.get('promotion_authority') is False and pair[0] == 'tiny-model' and pair in eligible:
        choices.append((eligible[pair], path, job))
assert choices, 'No committed tiny-model manifest survived the latest CPU screen.'
choices.sort(key=lambda x: (-x[0], str(x[1])))
_, JOB_MANIFEST, JOB = choices[0]
print('Selected:', JOB_MANIFEST, JOB['cpu_screen'])


## Rebuild the locked public corpus and run interactively
Set `RUN=True` only while actively using this notebook. Google Colab free runtimes are availability-dependent and are not used as unattended distributed workers.

In [ ]:
WORK = REPO / 'runs/colab-public-corpus'
LOCK = REPO / 'runs/colab-bootstrap-lock.json'
REBUILT = REPO / 'runs/colab-rebuilt-tokenizer.json'
shutil.copy2(REPO/'data/bootstrap-tokenizer-lock.json', LOCK)
subprocess.run([sys.executable,'-m','genesis_ai.bootstrap_corpus','--catalog','data/bootstrap-tokenizer-sources.json','--workspace',str(WORK.relative_to(REPO)),'--output',str(REBUILT.relative_to(REPO)),'--lock',str(LOCK.relative_to(REPO)),'--vocab-size','512'], check=True)
assert LOCK.read_bytes() == (REPO/'data/bootstrap-tokenizer-lock.json').read_bytes()
assert REBUILT.read_bytes() == (REPO/'tokenizers/genesis-v0.json').read_bytes()
CPU_SUMMARY = summary_path
RUN = False
OUTPUT_DIR = pathlib.Path('/content/genesis-ai-output')
if RUN:
    subprocess.run([sys.executable,'-m','genesis_ai.accelerator_job','run','--job',str(JOB_MANIFEST),'--cpu-summary',str(CPU_SUMMARY),'--platform','colab','--output-dir',str(OUTPUT_DIR)], check=True)
else:
    print('Ready. Set RUN=True while actively using Colab to execute the screened job.')
